# Lab 71: The Gestalt Closer

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 8, *The Social Network*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-71.ipynb)

**What you will do:** collect vividness ratings for the Kanizsa triangle with its wedges facing inward versus outward, and see whether a classic edge detector completes the illusory triangle the way your visual system does.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 71 you drew three Pac-Man shapes with their "mouths" facing inward, and a bright white
triangle appeared in the middle — the **Kanizsa triangle** — even though no triangle is
actually drawn. Figure 8.2 shows that rotating the same three shapes so their wedges face
outward removes the inward-pointing corners, and the illusory triangle vanishes.

## Record your results

Rate how vivid the illusory triangle looks (0 = no triangle at all, 10 = as solid as a real
printed triangle) for both versions of the figure, from yourself and a few friends.

In [ ]:
# example data: replace with yours
ratings = pd.DataFrame({
    "person": ["You", "Friend 1", "Friend 2", "Friend 3"],
    "inward_wedges_vividness": [8, 7, 9, 6],    # Figure 8.2(a): illusory triangle should appear
    "outward_wedges_vividness": [1, 0, 2, 1],   # Figure 8.2(b): illusion should vanish
})
display(ratings)
ratings.mean(numeric_only=True).plot.bar(color=["steelblue", "grey"], figsize=(5, 3.5))
plt.ylabel("mean vividness (0-10)"); plt.ylim(0, 10); plt.xticks(rotation=15); plt.show()

## Compare

Figure 8.2's caption states plainly that with the wedges facing outward, "the illusory
triangle vanishes." Check whether your ratings show that contrast.

In [ ]:
inward = ratings.inward_wedges_vividness.mean()
outward = ratings.outward_wedges_vividness.mean()
print(f"Inward-pointing wedges (illusory triangle): mean vividness {inward:.1f}/10")
print(f"Outward-pointing wedges (Figure 8.2b control): mean vividness {outward:.1f}/10")
if inward > outward:
    print("This matches the figure caption: the triangle 'vanishes' once there are no")
    print("inward-pointing corners to organise, confirming the illusion depends on the")
    print("arrangement, not on the Pac-Man shapes themselves.")
else:
    print("Unexpected -- check that the control figure really has its wedges rotated outward.")

## The AI side

The book links Gestalt closure to how AI systems go beyond their literal input. Below, the
classic **Canny edge detector** (Canny, 1986) is run on a Kanizsa figure generated from
scratch, to see whether a purely low-level, brightness-based algorithm "sees" the illusory
triangle's edges the way you do.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import math
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

# Draw a Kanizsa triangle from scratch: three Pac-Man shapes with mouths pointing at a
# shared centre, exactly like Figure 8.2(a).
size = 300
img = Image.new("L", (size, size), 255)
draw = ImageDraw.Draw(img)
centers = [(90, 90), (210, 90), (150, 220)]
centroid = (sum(c[0] for c in centers) / 3, sum(c[1] for c in centers) / 3)
radius, mouth_halfwidth = 40, 35
for cx, cy in centers:
    draw.ellipse([cx - radius, cy - radius, cx + radius, cy + radius], fill=0)
    angle = math.degrees(math.atan2(centroid[1] - cy, centroid[0] - cx))
    draw.pieslice([cx - radius, cy - radius, cx + radius, cy + radius],
                  angle - mouth_halfwidth, angle + mouth_halfwidth, fill=255)

edges = cv2.Canny(np.array(img), 50, 150)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("Kanizsa figure"); axes[0].axis("off")
axes[1].imshow(edges, cmap="gray"); axes[1].set_title("Canny edges (Canny, 1986)"); axes[1].axis("off")
plt.tight_layout(); plt.show()
print("Canny only marks edges where pixel brightness actually changes -- the three notches --")
print("not the illusory triangle's sides, which have no brightness change there to detect.")

A classic edge detector like Canny has no closure mechanism at all: it responds only to
local brightness gradients, so the illusory triangle's "edges" (which are not brightness
changes, only implied ones) are simply invisible to it. Modern segmentation networks can
sometimes complete boundaries like this because they have learned, from millions of labelled
images, statistical priors about how objects usually continue behind gaps — closer to your
own learned prior about how surfaces and shapes behave, but built from very different
training data and with no felt experience of a bright surface appearing.

## Questions to think about

1. The book argues that Gestalt principles reflect the statistical structure of the natural world, not arbitrary aesthetic taste. What is the most common real-world situation you can think of where three separate objects really would imply a hidden fourth shape behind them?
2. The LOC (lateral occipital complex) responds to the Kanizsa triangle almost as strongly as to a real, physically drawn one. If you had brain-imaging data instead of a vividness rating, what would "almost as strongly" need to look like numerically to convince you the completion is a genuine perceptual construction, not just a figure of speech?
3. Canny's edges mark only real brightness changes. Try increasing the two threshold numbers in the code (50 and 150) — does the notch at each Pac-Man's mouth ever disappear, and what does that tell you about the difference between an edge existing and an edge being detected?
4. JPEG compression relies on your own closure mechanism to reconstruct detail it throws away. Look closely at a heavily compressed photo you have on your phone -- can you find a region where you now suspect your brain is filling in more than the file actually contains?

## Challenge

Draw a Kanizsa-style figure using a different inducing shape instead of Pac-Man discs — for
example, four right-angle corners arranged to imply a square, or an outline letter with large
gaps. Rate how vividly the illusory shape appears compared with the classic triangle, and note
whether some inducing shapes produce a much weaker illusion than others.

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-71.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")